In [1]:
import json
import re
import sys
from datetime import datetime

# --------------------------------------------------------------------------
# Schema: for every segment, its elements IN ORDER.
#   (key, None)         -> simple element
#   (key, [comp, ...])  -> composite element with its components in order
# To support another message/segment, just add it here.
# --------------------------------------------------------------------------
SCHEMA = {
    "UNB": [("C_S001", ["E_0001", "E_0002"]),
            ("C_S002", ["E_0004", "E_0007"]),
            ("C_S003", ["E_0010", "E_0007"]),
            ("C_S004", ["E_0017", "E_0019"]),
            ("E_0020", None)],
    "UNH": [("E_0062", None),
            ("C_S009", ["E_0065", "E_0052", "E_0054", "E_0051", "E_0057"])],
    "BGM": [("C_002", ["E_1001", "E_1131", "E_3055", "E_1000"]),
            ("E_1004", None), ("E_1225", None)],
    "FTX": [("E_4451", None), ("E_4453", None),
            ("C_107", ["E_4441", "E_1131", "E_3055"]),
            ("C_108", ["E_4440"])],
    "TDT": [("E_8051", None), ("E_8028", None),
            ("C_220", ["E_8067", "E_8066"]),
            ("C_001", ["E_8179", "E_1131", "E_3055", "E_8178"]),
            ("C_040", ["E_3127", "E_1131", "E_3055", "E_3126"]),
            ("E_8101", None),
            ("C_401", ["E_8457", "E_8459", "E_7130"]),
            ("C_222", ["E_8213", "E_1131", "E_3055", "E_8212", "E_8453"])],
    "RFF": [("C_506", ["E_1153", "E_1154", "E_1156", "E_4000", "E_1060"])],
    "LOC": [("E_3227", None),
            ("C_517", ["E_3225", "E_1131", "E_3055", "E_3224"]),
            ("C_519", ["E_3223", "E_1131", "E_3055", "E_3222"]),
            ("C_553", ["E_3233", "E_1131", "E_3055", "E_3232"])],
    "DTM": [("C_507", ["E_2005", "E_2380", "E_2379"])],
    "NAD": [("E_3035", None),
            ("C_082", ["E_3039", "E_1131", "E_3055"])],
    "EQD": [("E_8053", None),
            ("C_237", ["E_8260", "E_1131", "E_3055", "E_3207"]),
            ("C_224", ["E_8155", "E_1131", "E_3055", "E_8154"]),
            ("E_8077", None), ("E_8249", None), ("E_8169", None)],
    "MEA": [("E_6311", None),
            ("C_502", ["E_6313", "E_6321", "E_6155", "E_6154"]),
            ("C_174", ["E_6411", "E_6314", "E_6162", "E_6152", "E_6432"])],
    "TMP": [("E_6245", None), ("C_239", ["E_6246", "E_6411"])],
    "SEL": [("E_9308", None),
            ("C_215", ["E_9303", "E_1131", "E_3055"]),
            ("E_4517", None)],
    "CNT": [("C_270", ["E_6069", "E_6066", "E_6411"])],
    "UNT": [("E_0074", None), ("E_0062", None)],
    "UNZ": [("E_0036", None), ("E_0020", None)],
}

In [2]:
# Segments that can occur many times -> always emitted as a JSON list
REPEATING = {"RFF", "LOC", "DTM", "NAD", "EQD", "MEA", "TMP", "SEL"}

# DTM format code -> Python strptime format
DATE_FORMATS = {"101": "%y%m%d", "102": "%Y%m%d", "201": "%y%m%d%H%M",
                "203": "%Y%m%d%H%M", "204": "%Y%m%d%H%M%S"}


# parsing
def split_unescaped(text, sep, release="?"):
    """Split on `sep`, ignoring separators escaped with the release char."""
    parts, cur, i = [], [], 0
    while i < len(text):
        c = text[i]
        if c == release and i + 1 < len(text):
            cur.append(text[i:i + 2])      # keep escape pair, unescape later
            i += 2
            continue
        if c == sep:
            parts.append("".join(cur))
            cur = []
        else:
            cur.append(c)
        i += 1
    parts.append("".join(cur))
    return parts


def unescape(value, release="?"):
    return re.sub(re.escape(release) + r"(.)", r"\1", value)


def read_segments(text):
    """Return a list of raw segment strings, honouring an optional UNA."""
    comp, elem, _dec, rel, _rsv, term = ":", "+", ".", "?", " ", "'"
    text = text.lstrip("\ufeff \r\n\t")
    if text.startswith("UNA"):
        comp, elem, _dec, rel, _rsv, term = text[3:9]
        text = text[9:]
    segs = [s.strip("\r\n \t") for s in split_unescaped(text, term, rel)]
    return [s for s in segs if s], comp, elem, rel


# Helper functions 
def to_iso(value, fmt_code):
    fmt = DATE_FORMATS.get(fmt_code)
    if not fmt or not value:
        return None
    try:
        return datetime.strptime(value, fmt).isoformat()
    except ValueError:
        return None


def stowage_position(value):
    """0300482 -> bay 030, row 04, tier 82 (also 6-digit 2/2/2 form)."""
    if value.isdigit() and len(value) == 7:
        n = (3, 2, 2)
    elif value.isdigit() and len(value) == 6:
        n = (2, 2, 2)
    else:
        return None
    b, r, t = value[:n[0]], value[n[0]:n[0] + n[1]], value[n[0] + n[1]:]
    return {"E_OriginalValue": value, "E_Bay": b, "E_Row": r, "E_Tier": t}


# Segment -> dict
def parse_segment(raw, comp_sep, elem_sep, rel):
    parts = split_unescaped(raw, elem_sep, rel)
    tag, elements = parts[0], parts[1:]
    schema = SCHEMA.get(tag)
    if schema is None:                      # unknown segment: keep raw values
        return tag, {"_raw": [unescape(e, rel) for e in elements]}
    if len(elements) > len(schema):
        print(f"warning: {tag} has {len(elements)} elements, schema knows "
              f"{len(schema)} -> extra ignored", file=sys.stderr)

    out = {}
    for i, (key, comps) in enumerate(schema):
        value = elements[i] if i < len(elements) else ""
        if comps is None:
            out[key] = unescape(value, rel)
        else:
            cvals = split_unescaped(value, comp_sep, rel)
            out[key] = {c: unescape(cvals[j], rel) if j < len(cvals) else ""
                        for j, c in enumerate(comps)}

    # helper fields
    if tag == "UNB":
        d = out["C_S004"]
        iso = to_iso("20" + d["E_0017"] + d["E_0019"], "203") \
            if len(d["E_0017"]) == 6 else None
        if iso:
            d["E_FormattedDateTime"] = iso
    elif tag == "DTM":
        d = out["C_507"]
        iso = to_iso(d["E_2380"], d["E_2379"])
        if iso:
            d["E_FormattedDateTime"] = iso
    elif tag == "LOC" and out["E_3227"] == "147":
        sp = stowage_position(out["C_517"]["E_3225"])
        if sp:
            out["C_StowagePosition"] = sp
    return tag, out


def edi_to_json(text):
    segments, comp_sep, elem_sep, rel = read_segments(text)
    result = {}
    for raw in segments:
        tag, data = parse_segment(raw, comp_sep, elem_sep, rel)
        if tag in REPEATING:
            result.setdefault(tag, []).append(data)
        else:
            result[tag] = data
    return result



In [4]:
file_path = 'source.EDI'

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

infoset = edi_to_json(text)

out = json.dumps(infoset, indent=2, ensure_ascii=False)
print(out)

with open("result.json", "w", encoding="utf-8") as f_out:
    f_out.write(out)

{
  "UNB": {
    "C_S001": {
      "E_0001": "UNOA",
      "E_0002": "2"
    },
    "C_S002": {
      "E_0004": "TMT",
      "E_0007": ""
    },
    "C_S003": {
      "E_0010": "APA",
      "E_0007": ""
    },
    "C_S004": {
      "E_0017": "260402",
      "E_0019": "1948",
      "E_FormattedDateTime": "2026-04-02T19:48:00"
    },
    "E_0020": "20260402470520"
  },
  "UNH": {
    "E_0062": "26040247052001",
    "C_S009": {
      "E_0065": "COARRI",
      "E_0052": "D",
      "E_0054": "95B",
      "E_0051": "UN",
      "E_0057": "SMDG16"
    }
  },
  "BGM": {
    "C_002": {
      "E_1001": "270",
      "E_1131": "",
      "E_3055": "",
      "E_1000": ""
    },
    "E_1004": "26040247052001",
    "E_1225": "9"
  },
  "FTX": {
    "E_4451": "OSI",
    "E_4453": "",
    "C_107": {
      "E_4441": "L",
      "E_1131": "",
      "E_3055": ""
    },
    "C_108": {
      "E_4440": ""
    }
  },
  "TDT": {
    "E_8051": "20",
    "E_8028": "214",
    "C_220": {
      "E_8067": "1",
      "E